# 11 · Sensibilidad de alcance: Bogotá–Región vs. Bogotá D. C. interno

**Objetivo.** Determinar si la brecha mujer–hombre en tiempo medio de primera llegada (FPT) depende de analizar toda la Bogotá–Región observada por la EODH 2023 o únicamente viajes y residencias internos de Bogotá D. C.

Esta es la **Compuerta 12**. El caso distrital reconstruye de manera independiente el kernel, los conjuntos `core_50` y las distribuciones residenciales. Además, se descompone la contribución regional por municipio y se hace una sensibilidad *leave-one-municipality-out* sobre el soporte residencial de origen. Esta última no elimina el municipio del kernel y, por tanto, es una prueba de influencia de los orígenes, no una simulación de cierre territorial.


In [ ]:
%pip -q install pandas pyarrow numpy scipy matplotlib seaborn geopandas networkx


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import os
import re
import shutil
import unicodedata
import zipfile

import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path("/content/colombia-stochastic-access")
RAW = ROOT / "data/raw/bogota_em2023"
INTERIM = ROOT / "data/interim/bogota_em2023_gate12"
OUT = ROOT / "outputs/bogota_em2023/gate12"
DRIVE_BASE = Path("/content/drive/MyDrive/colombia-stochastic-access-data/bogota_em2023")
for directory in (RAW, INTERIM, OUT):
    directory.mkdir(parents=True, exist_ok=True)

SOURCE_URL = "https://observatorio.movilidadbogota.gov.co/sites/default/files/2025-09/05_Base%20datos%20procesada%20EODH.zip"
ZONING_URL = "https://observatorio.movilidadbogota.gov.co/sites/default/files/2025-09/03_Zonificacion%20EODH%20%281%29.zip"
PURPOSES = ("employment", "education", "health")
PRIMARY_TAU = 20.0
BOOTSTRAP_REPLICATES = 200
RANDOM_SEED = 202312
MIN_DESTINATION_SAMPLE = 5
MIN_DESTINATION_ESS = 3.0

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print(f"Entorno fuera de Colab o Drive no montado: {exc}")

GATE11_DIR = Path(os.environ.get("GATE11_DIR_OVERRIDE", DRIVE_BASE / "results/gate11"))
GATE11_PATH = GATE11_DIR / "gate_11.json"
REGION_POINT_PATH = GATE11_DIR / "final_female_male_point_estimates.csv"
REGION_INTERVAL_PATH = GATE11_DIR / "final_female_male_intervals.csv"
REGION_ORIGIN_PATH = GATE11_DIR / "female_male_origin_contributions.csv"
required = (GATE11_PATH, REGION_POINT_PATH, REGION_INTERVAL_PATH, REGION_ORIGIN_PATH)
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Faltan insumos de la Compuerta 11: " + ", ".join(missing))
gate11 = json.loads(GATE11_PATH.read_text(encoding="utf-8"))
gate11_status = gate11.get("gate", gate11)
if not bool(gate11_status.get("gate_11_passed", False)):
    raise RuntimeError("La Compuerta 11 no fue aprobada.")

print("Alcances: Bogotá–Región (Gate 11) y Bogotá D. C. interno (reestimado)")
print("Bootstrap distrital:", BOOTSTRAP_REPLICATES)
print("Salida:", OUT)


In [ ]:
def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def valid_zip(path):
    path = Path(path)
    return path.exists() and path.stat().st_size > 0 and zipfile.is_zipfile(path)


def upload_archive(destination, url, label):
    from google.colab import files
    print(f"Descargue y seleccione el ZIP de {label}:")
    print(url)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Suba exactamente un archivo ZIP.")
    name, content = next(iter(uploaded.items()))
    temporary = destination.with_suffix(".zip.upload")
    temporary.write_bytes(content)
    del content
    if not valid_zip(temporary):
        temporary.unlink(missing_ok=True)
        raise ValueError(f"{name} no es un ZIP válido.")
    temporary.replace(destination)
    return "manual_upload"


def ensure_archive(runtime_name, drive_name, url, label):
    runtime = RAW / runtime_name
    cached = DRIVE_BASE / drive_name
    if valid_zip(runtime):
        method = "runtime_cache"
    elif valid_zip(cached):
        shutil.copy2(cached, runtime)
        method = "google_drive_cache"
    else:
        method = upload_archive(runtime, url, label)
        DRIVE_BASE.mkdir(parents=True, exist_ok=True)
        shutil.copy2(runtime, cached)
    return runtime, method


def safe_extract_recursive(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    queue = [(Path(zip_path), destination)]
    visited = set()
    while queue:
        current, target = queue.pop(0)
        signature = (str(current.resolve()), str(target.resolve()))
        if signature in visited:
            continue
        visited.add(signature)
        root = target.resolve()
        with zipfile.ZipFile(current) as zipped:
            for member in zipped.infolist():
                resolved = (target / member.filename).resolve()
                if root != resolved and root not in resolved.parents:
                    raise ValueError(f"Ruta insegura en ZIP: {member.filename}")
            zipped.extractall(target)
        for nested in target.rglob("*.zip"):
            nested_target = nested.parent / nested.stem
            nested_signature = (str(nested.resolve()), str(nested_target.resolve()))
            if nested_signature not in visited:
                queue.append((nested, nested_target))


survey_archive, survey_method = ensure_archive(
    "processed_edoh.zip", "processed_edoh.zip", SOURCE_URL, "microdatos procesados"
)
zoning_archive, zoning_method = ensure_archive(
    "zoning_edoh.zip", "zoning_edoh.zip", ZONING_URL, "zonificación"
)
safe_extract_recursive(survey_archive, INTERIM / "processed_edoh")
safe_extract_recursive(zoning_archive, INTERIM / "zoning_edoh")

source_manifest = pd.DataFrame([
    {"source_id": "processed_edoh", "url": SOURCE_URL, "path": str(survey_archive),
     "bytes": int(survey_archive.stat().st_size), "sha256": sha256(survey_archive),
     "retrieval_method": survey_method},
    {"source_id": "zoning_edoh", "url": ZONING_URL, "path": str(zoning_archive),
     "bytes": int(zoning_archive.stat().st_size), "sha256": sha256(zoning_archive),
     "retrieval_method": zoning_method},
])
source_manifest["generated_utc"] = datetime.now(timezone.utc).isoformat()
source_manifest.to_csv(OUT / "source_manifest.csv", index=False)
display(source_manifest)


In [ ]:
def normalize(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(char for char in value if not unicodedata.combining(char))
    return re.sub(r"[^a-z0-9]+", "_", value.lower().strip()).strip("_")


def clean_key(series):
    cleaned = series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    invalid = cleaned.map(normalize).isin(["", "nan", "none", "na", "no_aplica", "sin_informacion"])
    return cleaned.mask(invalid)


def canonical_utam(series):
    cleaned = clean_key(series)
    def canonical(value):
        if pd.isna(value):
            return pd.NA
        text = str(value).strip().upper().replace(" ", "")
        upr = re.fullmatch(r"UPR0*(\d+)", text)
        if upr:
            return f"UPR{int(upr.group(1)):04d}"
        utam = re.fullmatch(r"(?:UTAM)?0*(\d+)", text)
        if utam:
            return f"UTAM{int(utam.group(1)):03d}"
        return text
    return cleaned.map(canonical).astype("string")


def parse_number(series):
    raw = series.astype("string").str.strip()
    if float(raw.str.contains(",", regex=False, na=False).mean()) > 0.05:
        raw = raw.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(raw, errors="coerce")


def csv_format(path):
    for encoding in ("utf-8-sig", "latin-1"):
        try:
            with path.open("r", encoding=encoding, errors="strict") as handle:
                sample = handle.read(32768)
            return encoding, csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
        except (UnicodeDecodeError, csv.Error):
            continue
    raise ValueError(f"No fue posible detectar el formato: {path}")


def find_module(token, excluded=()):
    candidates = [
        path for path in (INTERIM / "processed_edoh").rglob("*.csv")
        if token in normalize(path.stem)
        and not any(item in normalize(path.stem) for item in excluded)
    ]
    if not candidates:
        raise FileNotFoundError(f"No se encontró el módulo CSV: {token}")
    return sorted(candidates, key=lambda path: (len(path.parts), len(path.name)))[0]


def read_selected(path, required):
    encoding, delimiter = csv_format(path)
    header = pd.read_csv(path, sep=delimiter, encoding=encoding, nrows=0)
    mapping = {normalize(column): column for column in header.columns}
    absent = sorted(set(required) - set(mapping))
    if absent:
        raise KeyError(f"Faltan columnas en {path.name}: {absent}")
    frame = pd.read_csv(path, sep=delimiter, encoding=encoding,
                        usecols=[mapping[column] for column in required],
                        dtype="string", low_memory=False)
    frame.columns = [normalize(column) for column in frame.columns]
    return frame


trips = read_selected(
    find_module("viaje", excluded=("etapa",)),
    ["key_hg", "utam_ori", "utam_des", "fexp_vj", "sexo", "motivo_viaje"],
)
persons = read_selected(
    find_module("persona"),
    ["key_hg", "cod_utam_hg", "fexp_per_5anos", "sexo"],
)
trips["household"] = clean_key(trips["key_hg"])
trips["utam_ori"] = canonical_utam(trips["utam_ori"])
trips["utam_des"] = canonical_utam(trips["utam_des"])
trips["weight"] = parse_number(trips["fexp_vj"])
trips["sex_group"] = trips["sexo"].map(normalize)
trips["purpose_normalized"] = trips["motivo_viaje"].map(normalize)
persons["household"] = clean_key(persons["key_hg"])
persons["residence_utam"] = canonical_utam(persons["cod_utam_hg"])
persons["weight"] = parse_number(persons["fexp_per_5anos"])
persons["sex_group"] = persons["sexo"].map(normalize)
trips = trips[
    trips[["household", "utam_ori", "utam_des"]].notna().all(axis=1)
    & trips["weight"].gt(0)
].reset_index(drop=True)
persons = persons[
    persons[["household", "residence_utam"]].notna().all(axis=1)
    & persons["weight"].gt(0)
].reset_index(drop=True)

PURPOSE_PATTERNS = {
    "employment": r"^a_trabajar$",
    "education": r"estudi|educa|coleg|univers|jardin|escuela",
    "health": r"salud|medic|hospital|clinic|consulta|terapia",
    "job_search": r"buscar_trabajo",
    "mobile_work": r"conduzco_vehiculo_como_forma_de_trabajo",
}
def classify_purpose(value):
    matches = [purpose for purpose, pattern in PURPOSE_PATTERNS.items() if re.search(pattern, value)]
    return "ambiguous" if len(matches) > 1 else (matches[0] if matches else "other")
trips["purpose_group"] = trips["purpose_normalized"].map(classify_purpose)

spatial_paths = sorted(
    list((INTERIM / "zoning_edoh").rglob("*.shp"))
    + list((INTERIM / "zoning_edoh").rglob("*.geojson"))
    + list((INTERIM / "zoning_edoh").rglob("*.gpkg"))
)
candidates = []
for path in spatial_paths:
    try:
        preview = gpd.read_file(path, rows=5)
        columns = {normalize(column): column for column in preview.columns}
        if "utam" in columns:
            candidates.append(path)
    except Exception:
        continue
if not candidates:
    raise FileNotFoundError("No se encontró capa espacial UTAM.")
utam_path = sorted(candidates, key=lambda path: ("utam2023" not in normalize(path.stem), len(path.parts)))[0]
utam_raw = gpd.read_file(utam_path)
column_map = {normalize(column): column for column in utam_raw.columns}
municipality_key = "munnombre" if "munnombre" in column_map else next(
    (key for key in column_map if "mun" in key and "nombre" in key), None
)
if municipality_key is None:
    raise KeyError("La capa UTAM no contiene nombre de municipio.")
utam_lookup = pd.DataFrame({
    "utam": canonical_utam(utam_raw[column_map["utam"]]),
    "municipality": utam_raw[column_map[municipality_key]].astype("string").str.strip(),
}).dropna(subset=["utam", "municipality"]).drop_duplicates("utam")
utam_lookup["municipality_normalized"] = utam_lookup["municipality"].map(normalize)
city_all_nodes = set(utam_lookup.loc[
    utam_lookup["municipality_normalized"].str.contains("bogota", na=False), "utam"
])
print("UTAM identificadas como Bogotá D. C.:", len(city_all_nodes))
print("Capa:", utam_path)


In [ ]:
city_trips_all = trips[
    trips["utam_ori"].isin(city_all_nodes) & trips["utam_des"].isin(city_all_nodes)
].copy()
city_persons_all = persons[persons["residence_utam"].isin(city_all_nodes)].copy()

graph = nx.DiGraph()
edge_pairs = city_trips_all.groupby(["utam_ori", "utam_des"], observed=True)["weight"].sum()
graph.add_weighted_edges_from((origin, destination, float(weight)) for (origin, destination), weight in edge_pairs.items())
components = sorted(nx.strongly_connected_components(graph), key=len, reverse=True)
if not components:
    raise RuntimeError("La red distrital no tiene componente fuertemente conexa.")
city_nodes_set = set(components[0])
city_nodes = sorted(city_nodes_set)
node_to_index = {node: index for index, node in enumerate(city_nodes)}
n_nodes = len(city_nodes)
city_trips = city_trips_all[
    city_trips_all["utam_ori"].isin(city_nodes_set)
    & city_trips_all["utam_des"].isin(city_nodes_set)
].reset_index(drop=True)
city_persons = city_persons_all[
    city_persons_all["residence_utam"].isin(city_nodes_set)
].reset_index(drop=True)

trip_weight_coverage = float(city_trips["weight"].sum() / city_trips_all["weight"].sum())
person_weight_coverage = float(city_persons["weight"].sum() / city_persons_all["weight"].sum())

origin_index = city_trips["utam_ori"].map(node_to_index).to_numpy(dtype=int)
destination_index = city_trips["utam_des"].map(node_to_index).to_numpy(dtype=int)
base_trip_weight = city_trips["weight"].to_numpy(dtype=float)
residence_index = city_persons["residence_utam"].map(node_to_index).to_numpy(dtype=int)
base_person_weight = city_persons["weight"].to_numpy(dtype=float)

W0 = np.zeros((n_nodes, n_nodes), dtype=float)
np.add.at(W0, (origin_index, destination_index), base_trip_weight)
row_sum = W0.sum(axis=1)
if np.any(row_sum <= 0):
    raise RuntimeError("La SCC distrital contiene filas sin salida.")
P0 = W0 / row_sum[:, None]

purpose_trips = city_trips[city_trips["purpose_group"].isin(PURPOSES)].copy()
purpose_trips["weight_sq"] = purpose_trips["weight"] ** 2
support = purpose_trips.groupby(["purpose_group", "utam_des"], observed=True).agg(
    sampled_trips=("weight", "size"), weighted_trips=("weight", "sum"),
    sum_weight_sq=("weight_sq", "sum"),
).reset_index()
support["effective_sample_size"] = support["weighted_trips"] ** 2 / support["sum_weight_sq"].replace(0, np.nan)
support["supported"] = support["sampled_trips"].ge(MIN_DESTINATION_SAMPLE) & support["effective_sample_size"].ge(MIN_DESTINATION_ESS)

target_rows = []
target_sets = {}
target_summary_rows = []
for purpose in PURPOSES:
    ranked = support[
        support["purpose_group"].eq(purpose) & support["supported"]
    ].sort_values("weighted_trips", ascending=False).copy()
    if ranked.empty:
        raise RuntimeError(f"Sin destinos soportados para {purpose}.")
    ranked["rank"] = np.arange(1, len(ranked) + 1)
    ranked["cumulative_supported_share"] = ranked["weighted_trips"].cumsum() / ranked["weighted_trips"].sum()
    crossing = np.flatnonzero(ranked["cumulative_supported_share"].to_numpy() >= 0.50)
    number = int(crossing[0] + 1) if len(crossing) else int(len(ranked))
    selected = ranked.head(number)
    target_sets[purpose] = np.array(sorted(node_to_index[node] for node in selected["utam_des"]), dtype=int)
    target_summary_rows.append({
        "scope": "bogota_dc_internal", "purpose_group": purpose,
        "target_definition": "core_50", "target_nodes": number,
        "selected_supported_share": float(selected["weighted_trips"].sum() / ranked["weighted_trips"].sum()),
    })
    for row in selected.itertuples(index=False):
        target_rows.append({
            "purpose_group": purpose, "target_definition": "core_50", "utam": row.utam_des,
            "rank": int(row.rank), "weighted_trips": float(row.weighted_trips),
            "effective_sample_size": float(row.effective_sample_size),
            "cumulative_supported_share": float(row.cumulative_supported_share),
        })

city_targets = pd.DataFrame(target_rows)
city_target_summary = pd.DataFrame(target_summary_rows)
scope_support = pd.DataFrame([{
    "city_utam_in_zoning": len(city_all_nodes), "largest_scc_nodes": n_nodes,
    "internal_trip_rows_before_scc": len(city_trips_all), "internal_trip_rows_in_scc": len(city_trips),
    "trip_weight_scc_coverage": trip_weight_coverage,
    "city_person_rows_before_scc": len(city_persons_all), "city_person_rows_in_scc": len(city_persons),
    "person_weight_scc_coverage": person_weight_coverage,
}])
city_targets.to_csv(OUT / "bogota_dc_internal_core50_targets.csv", index=False)
city_target_summary.to_csv(OUT / "bogota_dc_internal_target_summary.csv", index=False)
scope_support.to_csv(OUT / "bogota_dc_internal_scope_support.csv", index=False)
display(scope_support)
display(city_target_summary)


In [ ]:
SEX_LEVELS = ("mujer", "hombre")

def sex_mask(frame, level):
    return frame["sex_group"].eq(level).fillna(False).to_numpy(dtype=bool)

trip_masks = {level: sex_mask(city_trips, level) for level in SEX_LEVELS}
person_masks = {level: sex_mask(city_persons, level) for level in SEX_LEVELS}

def regularized_kernel(mask, row_multiplier=None):
    selected = np.flatnonzero(mask)
    original = base_trip_weight[selected]
    multiplicity = np.ones(len(selected)) if row_multiplier is None else row_multiplier[selected].astype(float)
    weights = original * multiplicity
    squared = original ** 2 * multiplicity
    origins, destinations = origin_index[selected], destination_index[selected]
    positive = weights > 0
    weights, squared = weights[positive], squared[positive]
    origins, destinations = origins[positive], destinations[positive]
    W = np.zeros((n_nodes, n_nodes), dtype=float)
    np.add.at(W, (origins, destinations), weights)
    out = W.sum(axis=1)
    sum_sq = np.bincount(origins, weights=squared, minlength=n_nodes)
    ess = np.divide(out ** 2, sum_sq, out=np.zeros(n_nodes), where=sum_sq > 0)
    empirical = np.zeros_like(W)
    observed = out > 0
    empirical[observed] = W[observed] / out[observed, None]
    omega = ess / (ess + PRIMARY_TAU)
    return omega[:, None] * empirical + (1.0 - omega[:, None]) * P0

def residential_distribution(mask, row_multiplier=None):
    selected = np.flatnonzero(mask)
    multiplicity = np.ones(len(selected)) if row_multiplier is None else row_multiplier[selected].astype(float)
    weights = base_person_weight[selected] * multiplicity
    mu = np.bincount(residence_index[selected], weights=weights, minlength=n_nodes).astype(float)
    if not np.isfinite(mu).all() or mu.sum() <= 0:
        raise RuntimeError("Distribución residencial vacía o no finita.")
    return mu / mu.sum()

def first_passage_by_origin(P, targets):
    target_mask = np.zeros(n_nodes, dtype=bool)
    target_mask[targets] = True
    transient = np.flatnonzero(~target_mask)
    Q = P[np.ix_(transient, transient)]
    mean = np.linalg.solve(np.eye(len(transient)) - Q, np.ones(len(transient)))
    full = np.zeros(n_nodes)
    full[transient] = mean
    return full

def conditioned_mu(mu, targets):
    result = mu.copy()
    result[targets] = 0.0
    if result.sum() <= 0:
        raise RuntimeError("Distribución condicionada vacía.")
    return result / result.sum()

def sex_gap_values(h_w, h_m, mu_w_raw, mu_m_raw, targets):
    mu_w = conditioned_mu(mu_w_raw, targets)
    mu_m = conditioned_mu(mu_m_raw, targets)
    mm, mw = float(mu_m @ h_m), float(mu_w @ h_m)
    wm, ww = float(mu_m @ h_w), float(mu_w @ h_w)
    residential = 0.5 * ((mw - mm) + (ww - wm))
    kernel = 0.5 * ((wm - mm) + (ww - mw))
    total = ww - mm
    return {
        "male_total_fpt": mm, "female_total_fpt": ww,
        "female_minus_male_total": total,
        "residential_contribution": residential, "kernel_contribution": kernel,
        "closure_error": total - residential - kernel,
    }

kernels = {level: regularized_kernel(trip_masks[level]) for level in SEX_LEVELS}
mus = {level: residential_distribution(person_masks[level]) for level in SEX_LEVELS}
point_rows = []
for purpose in PURPOSES:
    targets = target_sets[purpose]
    h_w = first_passage_by_origin(kernels["mujer"], targets)
    h_m = first_passage_by_origin(kernels["hombre"], targets)
    point_rows.append({
        "scope": "bogota_dc_internal", "purpose_group": purpose,
        **sex_gap_values(h_w, h_m, mus["mujer"], mus["hombre"], targets),
    })
city_point = pd.DataFrame(point_rows)
city_point.to_csv(OUT / "bogota_dc_internal_female_male_point_estimates.csv", index=False)
display(city_point)


In [ ]:
all_households = pd.Index(sorted(
    set(city_trips["household"].dropna().astype(str))
    | set(city_persons["household"].dropna().astype(str))
))
household_to_code = {household: index for index, household in enumerate(all_households)}
trip_household_codes = city_trips["household"].astype(str).map(household_to_code).to_numpy(dtype=int)
person_household_codes = city_persons["household"].astype(str).map(household_to_code).to_numpy(dtype=int)

rng = np.random.default_rng(RANDOM_SEED)
bootstrap_rows = []
for replicate in range(1, BOOTSTRAP_REPLICATES + 1):
    household_multiplier = rng.poisson(1, len(all_households))
    trip_multiplier = household_multiplier[trip_household_codes]
    person_multiplier = household_multiplier[person_household_codes]
    boot_kernels = {
        level: regularized_kernel(trip_masks[level], trip_multiplier) for level in SEX_LEVELS
    }
    boot_mus = {
        level: residential_distribution(person_masks[level], person_multiplier) for level in SEX_LEVELS
    }
    for purpose in PURPOSES:
        targets = target_sets[purpose]
        h_w = first_passage_by_origin(boot_kernels["mujer"], targets)
        h_m = first_passage_by_origin(boot_kernels["hombre"], targets)
        bootstrap_rows.append({
            "replicate": replicate, "scope": "bogota_dc_internal", "purpose_group": purpose,
            **sex_gap_values(h_w, h_m, boot_mus["mujer"], boot_mus["hombre"], targets),
        })
    if replicate % 25 == 0:
        print(f"Bootstrap distrital {replicate}/{BOOTSTRAP_REPLICATES}")

city_boot = pd.DataFrame(bootstrap_rows)
city_boot.to_parquet(OUT / "bogota_dc_internal_household_bootstrap.parquet", index=False)
effects = ["female_minus_male_total", "residential_contribution", "kernel_contribution"]
interval_rows = []
for purpose in PURPOSES:
    frame = city_boot[city_boot["purpose_group"].eq(purpose)]
    point_row = city_point.set_index("purpose_group").loc[purpose]
    for effect in effects:
        values = frame[effect]
        interval_rows.append({
            "scope": "bogota_dc_internal", "purpose_group": purpose, "effect": effect,
            "point_estimate": float(point_row[effect]), "bootstrap_mean": float(values.mean()),
            "bootstrap_sd": float(values.std()), "ci_lower": float(values.quantile(0.025)),
            "ci_upper": float(values.quantile(0.975)),
        })
city_intervals = pd.DataFrame(interval_rows)

simultaneous_rows = []
for effect in effects:
    wide = city_boot.pivot(index="replicate", columns="purpose_group", values=effect).reindex(columns=PURPOSES)
    point_values = city_point.set_index("purpose_group")[effect].reindex(PURPOSES)
    se = wide.std(axis=0).replace(0, np.nan)
    standardized = wide.subtract(point_values, axis=1).abs().divide(se, axis=1)
    critical = float(standardized.max(axis=1).quantile(0.95))
    for purpose in PURPOSES:
        estimate, standard_error = float(point_values[purpose]), float(se[purpose])
        simultaneous_rows.append({
            "purpose_group": purpose, "effect": effect, "max_t_critical": critical,
            "simultaneous_ci_lower": estimate - critical * standard_error,
            "simultaneous_ci_upper": estimate + critical * standard_error,
        })
city_intervals = city_intervals.merge(pd.DataFrame(simultaneous_rows), on=["purpose_group", "effect"])
city_intervals.to_csv(OUT / "bogota_dc_internal_female_male_intervals.csv", index=False)

region_point = pd.read_csv(REGION_POINT_PATH).assign(scope="bogota_region")
region_intervals = pd.read_csv(REGION_INTERVAL_PATH).assign(scope="bogota_region")
scope_points = pd.concat([
    region_point[["scope", "purpose_group", "male_total_fpt", "female_total_fpt",
                  "female_minus_male_total", "residential_contribution", "kernel_contribution", "closure_error"]],
    city_point,
], ignore_index=True)
scope_intervals = pd.concat([
    region_intervals[["scope", "purpose_group", "effect", "point_estimate", "bootstrap_mean",
                      "bootstrap_sd", "ci_lower", "ci_upper", "max_t_critical",
                      "simultaneous_ci_lower", "simultaneous_ci_upper"]],
    city_intervals,
], ignore_index=True)
scope_points.to_csv(OUT / "scope_comparison_point_estimates.csv", index=False)
scope_intervals.to_csv(OUT / "scope_comparison_intervals.csv", index=False)
display(scope_points)


In [ ]:
region_origin = pd.read_csv(REGION_ORIGIN_PATH, dtype={"utam": "string"})
region_origin["utam"] = canonical_utam(region_origin["utam"])
region_origin = region_origin.merge(utam_lookup[["utam", "municipality"]], on="utam", how="left")
municipality_mapping_coverage = float(region_origin["municipality"].notna().mean())
region_origin["municipality"] = region_origin["municipality"].fillna("<unmapped>")

municipality_contributions = region_origin.groupby(
    ["purpose_group", "municipality"], observed=True
).agg(
    residential_contribution=("residential_contribution", "sum"),
    kernel_contribution=("kernel_contribution", "sum"),
    total_contribution=("total_contribution", "sum"),
    female_residential_probability=("female_residential_probability", "sum"),
    male_residential_probability=("male_residential_probability", "sum"),
).reset_index()
municipality_contributions.to_csv(OUT / "region_municipality_origin_contributions.csv", index=False)

leave_rows = []
for purpose in PURPOSES:
    frame = region_origin[region_origin["purpose_group"].eq(purpose)].copy()
    baseline = region_point.set_index("purpose_group").loc[purpose]
    for municipality in sorted(frame["municipality"].unique()):
        keep = ~frame["municipality"].eq(municipality)
        mu_w = frame.loc[keep, "female_residential_probability"].to_numpy(dtype=float)
        mu_m = frame.loc[keep, "male_residential_probability"].to_numpy(dtype=float)
        if mu_w.sum() <= 0 or mu_m.sum() <= 0:
            continue
        mu_w, mu_m = mu_w / mu_w.sum(), mu_m / mu_m.sum()
        h_w = frame.loc[keep, "female_fpt_steps"].to_numpy(dtype=float)
        h_m = frame.loc[keep, "male_fpt_steps"].to_numpy(dtype=float)
        mm, mw, wm, ww = float(mu_m @ h_m), float(mu_w @ h_m), float(mu_m @ h_w), float(mu_w @ h_w)
        residence = 0.5 * ((mw - mm) + (ww - wm))
        kernel = 0.5 * ((wm - mm) + (ww - mw))
        gap = ww - mm
        leave_rows.append({
            "purpose_group": purpose, "excluded_origin_municipality": municipality,
            "female_residential_mass_removed": float(frame.loc[~keep, "female_residential_probability"].sum()),
            "male_residential_mass_removed": float(frame.loc[~keep, "male_residential_probability"].sum()),
            "female_minus_male_total": gap, "residential_contribution": residence,
            "kernel_contribution": kernel, "delta_gap_vs_region": gap - float(baseline["female_minus_male_total"]),
        })
leave_one_out = pd.DataFrame(leave_rows)
leave_one_out.to_csv(OUT / "region_origin_leave_one_municipality_out.csv", index=False)
display(municipality_contributions.sort_values(
    ["purpose_group", "kernel_contribution"], ascending=[True, False]
).groupby("purpose_group", observed=True).head(8))
display(leave_one_out.reindex(leave_one_out["delta_gap_vs_region"].abs().sort_values(ascending=False).index).head(15))


In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
plot_data = scope_intervals[scope_intervals["effect"].eq("female_minus_male_total")].copy()
fig, axis = plt.subplots(figsize=(9.2, 5.4))
palette = {"bogota_region": "#3B6FB6", "bogota_dc_internal": "#D96B3B"}
for offset, scope in zip((-0.13, 0.13), ("bogota_region", "bogota_dc_internal")):
    subset = plot_data[plot_data["scope"].eq(scope)].set_index("purpose_group").reindex(PURPOSES)
    x = np.arange(len(PURPOSES)) + offset
    y = subset["point_estimate"].to_numpy(dtype=float)
    lower = y - subset["simultaneous_ci_lower"].to_numpy(dtype=float)
    upper = subset["simultaneous_ci_upper"].to_numpy(dtype=float) - y
    axis.errorbar(x, y, yerr=[lower, upper], fmt="o", capsize=4,
                  label=scope.replace("_", " ").title(), color=palette[scope])
axis.axhline(0, color="black", linewidth=0.8)
axis.set_xticks(np.arange(len(PURPOSES)))
axis.set_xticklabels([purpose.title() for purpose in PURPOSES])
axis.set_ylabel("Female − male mean FPT (steps)")
axis.set_xlabel("Destination purpose")
axis.legend()
fig.tight_layout()
fig.savefig(OUT / "figure_scope_comparison_sex_gap.png", dpi=240, bbox_inches="tight")
plt.show()

influence = leave_one_out.copy()
top_names = influence.groupby("excluded_origin_municipality", observed=True)["delta_gap_vs_region"].apply(
    lambda values: values.abs().max()
).nlargest(10).index
influence = influence[influence["excluded_origin_municipality"].isin(top_names)]
fig, axis = plt.subplots(figsize=(10, 6))
sns.barplot(data=influence, y="excluded_origin_municipality", x="delta_gap_vs_region",
            hue="purpose_group", orient="h", ax=axis)
axis.axvline(0, color="black", linewidth=0.8)
axis.set_xlabel("Change in regional female − male gap after excluding origin support")
axis.set_ylabel("Excluded municipality")
fig.tight_layout()
fig.savefig(OUT / "figure_municipality_origin_influence.png", dpi=240, bbox_inches="tight")
plt.show()


In [ ]:
report_point = pd.DataFrame(gate11.get("sex_point_estimates", []))
replication_columns = ["female_minus_male_total", "residential_contribution", "kernel_contribution"]
if report_point.empty:
    region_replication_error = np.inf
else:
    comparison = region_point.merge(report_point, on="purpose_group", suffixes=("_csv", "_json"))
    region_replication_error = max(
        float((comparison[f"{column}_csv"] - comparison[f"{column}_json"]).abs().max())
        for column in replication_columns
    )

thresholds = {
    "minimum_scc_trip_weight_coverage": 0.95,
    "minimum_scc_person_weight_coverage": 0.95,
    "minimum_core_target_nodes": 3,
    "maximum_closure_error": 1e-10,
    "minimum_municipality_mapping_coverage": 0.99,
    "maximum_gate11_replication_error": 1e-9,
}
checks = {
    "gate11_dependency_valid": bool(gate11_status.get("gate_11_passed", False)),
    "region_results_reproduced": bool(region_replication_error <= thresholds["maximum_gate11_replication_error"]),
    "district_kernel_row_stochastic": bool(np.allclose(P0.sum(axis=1), 1.0, atol=1e-10)),
    "district_scc_trip_coverage_valid": bool(trip_weight_coverage >= thresholds["minimum_scc_trip_weight_coverage"]),
    "district_scc_person_coverage_valid": bool(person_weight_coverage >= thresholds["minimum_scc_person_weight_coverage"]),
    "district_targets_valid": bool(city_target_summary["target_nodes"].ge(thresholds["minimum_core_target_nodes"]).all()),
    "district_point_estimates_finite": bool(np.isfinite(city_point.select_dtypes(include="number").to_numpy()).all()),
    "district_decomposition_identity_valid": bool(city_point["closure_error"].abs().max() <= thresholds["maximum_closure_error"]),
    "district_bootstrap_200_complete": bool(
        city_boot["replicate"].nunique() == BOOTSTRAP_REPLICATES
        and len(city_boot) == BOOTSTRAP_REPLICATES * len(PURPOSES)
    ),
    "district_intervals_finite": bool(np.isfinite(city_intervals[[
        "ci_lower", "ci_upper", "simultaneous_ci_lower", "simultaneous_ci_upper"
    ]].to_numpy()).all()),
    "municipality_mapping_valid": bool(municipality_mapping_coverage >= thresholds["minimum_municipality_mapping_coverage"]),
    "leave_one_municipality_out_complete": bool(
        set(PURPOSES).issubset(set(leave_one_out["purpose_group"]))
        and np.isfinite(leave_one_out.select_dtypes(include="number").to_numpy()).all()
    ),
}
checks["gate_12_passed"] = bool(all(checks.values()))

report = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "estimand": "scope sensitivity of female-minus-male mean FPT decomposition",
    "scope_definitions": {
        "bogota_region": "validated Gate 11 network for all EODH study-region trips and residences",
        "bogota_dc_internal": "trips with origin and destination in Bogotá D.C. UTAM; residences in Bogotá D.C.; largest directed SCC",
    },
    "district_specification": {
        "target_definition": "scope-specific core_50", "conditioning": "outside_target",
        "regularization_tau": PRIMARY_TAU, "bootstrap_replicates": BOOTSTRAP_REPLICATES,
        "bootstrap_method": "joint Poisson(1) household cluster bootstrap",
    },
    "origin_influence_note": "Leave-one-municipality-out renormalizes residential starting support while retaining the regional transition kernel.",
    "scope_support": json.loads(scope_support.to_json(orient="records"))[0],
    "district_target_summary": json.loads(city_target_summary.to_json(orient="records")),
    "scope_point_estimates": json.loads(scope_points.to_json(orient="records")),
    "municipality_mapping_coverage": municipality_mapping_coverage,
    "thresholds": thresholds,
    "gate": checks,
}
(OUT / "gate_12.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8"
)
pd.Series(checks, name="result").to_csv(OUT / "gate_12_checks.csv")
display(pd.Series(checks, name="result"))

persistent_out = DRIVE_BASE / "results/gate12"
persistent_out.mkdir(parents=True, exist_ok=True)
for artifact in OUT.iterdir():
    if artifact.is_file():
        shutil.copy2(artifact, persistent_out / artifact.name)

print("Compuerta 12:", "APROBADA" if checks["gate_12_passed"] else "NO APROBADA")
print("Copia persistente:", persistent_out)


## Interpretación antes de escribir el paper

- Si los signos, magnitudes y conclusiones inferenciales coinciden en ambos alcances, el resultado principal es robusto y Bogotá–Región puede mantenerse como caso principal, con Bogotá D. C. interno como sensibilidad.
- Si cambian materialmente, el alcance territorial es parte del resultado: deben reportarse ambos estimandos y evitar llamar “Bogotá” al estimando regional sin aclaración.
- La tabla municipal identifica de dónde proviene la contribución regional. El *leave-one-out* evalúa influencia de la mezcla residencial; no estima qué ocurriría si desapareciera la red o la movilidad de un municipio.
